# 02 — Features · Crowdfund Compass (Group 11)
Goal: turn the merged table into 35 features in 6 groups, using only what a founder knows **before launch**, and look at each group's raw relation to success.

Workflow: 1) Setup → 2) Import → 3) Build features → 4) Check the before-launch rule → 5) Signal per feature → 6) One chart per group → 7) Conclusions.

*AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup
`features.py` is our shared feature module (the app will use the same file). Put it in the same Drive folder as this notebook.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
!pip -q install vaderSentiment        # sentiment word list (beyond class)

# from google.colab import drive; drive.mount('/content/drive')
project = 'drive/My Drive/crowdfund_compass'      # folder with features.py
path = 'drive/My Drive/ks_scrapes/merged_v2'      # folder with kickstarter_merged_v2_part1..8.parquet
sys.path.insert(0, project)
import features as F

## 2) Import the data

In [ ]:
raw = F.load_merged(path)
print(len(raw), 'projects')

## 3) Build the features
One call builds everything (about 2 minutes). What it does, group by group:
- **Ambition:** goal in USD, goal compared with the category's median goal, goal per day, duration, round-number goal.
- **Timing:** launch month and weekday, campaign runs over Christmas, preparation time.
- **Pitch:** length of title and blurb, digits/symbols/capitals in the title, readability, sentiment, four keyword flags.
- **Founder:** number and success of the creator's earlier campaigns.
- **Presentation:** video, pre-launch page (known only for recent projects; used in the modern model only).
- **Market:** category, country, the category's recent funding rate, trend and size.

Dropped after this notebook's first run (16.11): three crowding features and city size. Inside a category they had no signal (AUC 0.49–0.51), and crowding mostly reflected which categories the scraper covers well.

History rule: category norms use campaigns that ended in the 12 months before the first day of the launch month; creator history uses only campaigns that ended before the launch itself.

In [ ]:
X, tables = F.build_features(raw)
features = [f for group in F.GROUPS.values() for f in group]
print(len(features), 'features in', len(F.GROUPS), 'groups')
for group, cols in F.GROUPS.items():
    print(f'{group:13s}', cols)

## 4) Check the before-launch rule
We recompute the creator history for 200 random projects in a slow, obvious way and compare it with the module. Any mismatch would mean a bug or a leak.

In [ ]:
hist = raw[raw.state.isin(['successful', 'failed', 'canceled', 'suspended']) & (raw.launched_at > 1.2e9)]
mismatch = 0
for r in X[X.creator_prior_n > 0].sample(200, random_state=1).itertuples():
    earlier = hist[(hist.creator_id == r.creator_id) & (hist.deadline < r.launched_at)]   # ended BEFORE this launch
    mismatch += (len(earlier) != r.creator_prior_n) or ((earlier.state == 'successful').sum() != r.creator_prior_funded)
print('mismatches:', mismatch, 'of 200')

## 5) Signal per feature
For launches 2023–2024 (the years where every feature is known) we measure how well each feature **alone** separates funded from failed campaigns. AUC 0.5 = no signal; further from 0.5 = stronger. Because our sample over-represents funded campaigns differently in each category, we also compute the AUC **inside each parent category** and **inside each category**, then average.

In [ ]:
E = X[X.launch_year.isin([2023, 2024]) & X.has_video.notna()].copy()
print(len(E), 'campaigns | funded in sample:', round(E.success.mean(), 3))

def auc_within(df, feature, by=None):
    if by is None:
        v = df[[feature, 'success']].dropna()
        return roc_auc_score(v.success, v[feature])
    total = weight = 0
    for _, g in df.groupby(by):
        v = g[[feature, 'success']].dropna()
        if v.success.nunique() == 2 and v[feature].nunique() > 1:
            total += roc_auc_score(v.success, v[feature]) * len(v); weight += len(v)
    return total / weight

rows = []
for group, cols in F.GROUPS.items():
    for f in cols:
        if f in ('cat_id', 'parent_category', 'country'): continue      # labels, not numbers
        rows.append((group, f, auc_within(E, f), auc_within(E, f, 'parent_category'), auc_within(E, f, 'cat_id')))
signal = pd.DataFrame(rows, columns=['group', 'feature', 'AUC overall', 'AUC within parent', 'AUC within category']).round(3)
signal['strength'] = (signal['AUC within category'] - 0.5).abs()
print(signal.sort_values(['group', 'strength'], ascending=[True, False]).to_string(index=False))

**Interpretation:** the creator's track record, the goal, the preparation time and the pre-launch page carry signal even inside a category. The market features (category funding rate and size) look strongest overall but lose most of their signal inside a category: part of what they measure is which categories the scraper covers well, not real odds.

## 6) One chart per feature group
Each chart shows the share of campaigns funded **in our sample** (launches 2023–2024). These are associations, not causes.

In [ ]:
AVG = E.success.mean(); N = len(E)
ACC, GREY, INK, MUTE = '#0F7C82', '#C5CAD3', '#1B2A41', '#5B6676'
SUB = f'Share of campaigns funded, in our sample. Finished Kickstarter campaigns launched 2023–2024 (n = {N:,}).'
NOTE = 'Sample rates are higher than real Kickstarter rates (many failed campaigns are missing). Association, not cause.'
def bars(t,title,xlabel,fn,accent,horizontal=False,size=(9,5.2)):
    fig,ax=plt.subplots(figsize=size,dpi=150)
    cols=[ACC if i in accent else GREY for i in range(len(t))]
    v=t.funded.values*100; lab=[str(x) for x in t.index]
    if horizontal:
        y=np.arange(len(t))[::-1]; ax.barh(y,v,color=cols,height=.62)
        for yi,vi,ni in zip(y,v,t.n): ax.text(vi+1,yi,f'{vi:.0f}%  (n = {ni:,})',va='center',fontsize=10,color=INK,bbox=dict(fc='white',ec='none',pad=1.5),zorder=5)
        ax.set_yticks(y); ax.set_yticklabels(lab,fontsize=10.5,color=INK); ax.set_xlim(0,118); ax.set_xticks([0,25,50,75,100])
        ax.axvline(AVG*100,color=MUTE,lw=1,ls=(0,(4,3)),zorder=1); ax.text(AVG*100+1,len(t)-.42,f'all campaigns: {AVG*100:.0f}%',ha='left',fontsize=9,color=MUTE)
        ax.set_xlabel('Funded campaigns (%)',fontsize=10,color=MUTE); ax.set_ylabel(xlabel,fontsize=10,color=MUTE); ax.set_ylim(-.6,len(t)-.1)
    else:
        x=np.arange(len(t)); ax.bar(x,v,color=cols,width=.66)
        for xi,vi,ni in zip(x,v,t.n): ax.text(xi,vi+1.5,f'{vi:.0f}%',ha='center',fontsize=10.5,color=INK,bbox=dict(fc='white',ec='none',pad=1),zorder=5)
        ax.set_xticks(x); ax.set_xticklabels([f'{l}\nn = {ni:,}' for l,ni in zip(lab,t.n)],fontsize=9.5,color=INK); ax.set_ylim(0,108); ax.set_yticks([0,25,50,75,100])
        ax.axhline(AVG*100,color=MUTE,lw=1,ls=(0,(4,3)),zorder=1); ax.text(len(t)-.5,100,f'dashed line = all campaigns: {AVG*100:.0f}%',ha='right',fontsize=9,color=MUTE)
        ax.set_ylabel('Funded campaigns (%)',fontsize=10,color=MUTE); ax.set_xlabel(xlabel,fontsize=10,color=MUTE,labelpad=8)
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['left','bottom']: ax.spines[s].set_color('#D9DDE3')
    ax.tick_params(colors=MUTE,length=0)
    fig.text(.03,.955,title,fontsize=13.5,fontweight='bold',color=INK,ha='left',va='top',wrap=True)
    fig.text(.03,.86,SUB,fontsize=9.5,color=MUTE,ha='left',va='top')
    fig.text(.03,.02,NOTE,fontsize=8.5,color=MUTE,ha='left',va='bottom')
    fig.subplots_adjust(top=.78,bottom=.24 if not horizontal else .17,left=.33 if horizontal else .08,right=.97)
    plt.show(); print(t.round(3))
def tab(key): return E.groupby(key, observed=True).success.agg(n='size', funded='mean')

### 6.1 Ambition

In [ ]:
k=pd.cut(E.goal_ratio,[-9,-1,-.5,-.15,.15,.5,1,9],labels=['under 1/10','1/10 to 1/3','1/3 to 0.7','about the median','1.4 to 3×','3 to 10×','over 10×'])
t=tab(k); bars(t,f"Campaigns asking more than 10× their category's usual goal are funded {t.funded.iloc[-1]*100:.0f}% of the time,\nthose asking about the usual goal {t.funded.iloc[3]*100:.0f}%",'Goal compared with the median goal in the same category (campaigns ended in the 12 months before)','1_ambition',{6})

### 6.2 Timing

In [ ]:
k=pd.cut(E.log_prep_days,np.log10(np.array([0,1,3,7,14,30,90,365,1e5])+1),labels=['under 1 day','1–3 days','3–7 days','1–2 weeks','2–4 weeks','1–3 months','3–12 months','over 1 year'],include_lowest=True)
t=tab(k); bars(t,f"Campaigns launched within a day of creating the project page are funded {t.funded.iloc[0]*100:.0f}% of the time,\nthose prepared for 1–3 months {t.funded.iloc[5]*100:.0f}%",'Preparation time: from creating the project page to launch','2_timing',{0})

### 6.3 Pitch

In [ ]:
k=pd.cut(E.name_words,[0,3,5,7,9,30],labels=['1–3 words','4–5 words','6–7 words','8–9 words','10 or more'])
t=tab(k); bars(t,f"Campaigns with a title of 1–3 words are funded {t.funded.iloc[0]*100:.0f}% of the time,\nthose with 10 or more words {t.funded.iloc[-1]*100:.0f}%",'Length of the campaign title','3_pitch',{0})

### 6.4 Founder

In [ ]:
tr=np.select([E.creator_prior_n==0,(E.creator_prior_n>0)&E.creator_prior_rate.isna(),E.creator_prior_rate==0,E.creator_prior_rate<1],['First campaign','x','Earlier campaigns, none funded','Earlier campaigns, some funded'],'Earlier campaigns, all funded')
t=tab(pd.Series(tr,index=E.index)).drop('x').loc[['Earlier campaigns, all funded','Earlier campaigns, some funded','First campaign','Earlier campaigns, none funded']]
bars(t,f"First-time creators are funded {t.funded.iloc[2]*100:.0f}% of the time,\ncreators whose earlier campaigns were all funded {t.funded.iloc[0]*100:.0f}%","Creator's track record before this launch",'4_founder',{2},horizontal=True)

### 6.5 Presentation

In [ ]:
lab=np.select([(E.has_video==1)&(E.prelaunch_activated==1),(E.has_video==0)&(E.prelaunch_activated==1),(E.has_video==1)&(E.prelaunch_activated==0)],['Pre-launch page and video','Pre-launch page, no video','Video, no pre-launch page'],'Neither')
t=tab(pd.Series(lab,index=E.index)).loc[['Pre-launch page and video','Pre-launch page, no video','Video, no pre-launch page','Neither']]
bars(t,f"Campaigns with a pre-launch page are funded {E[E.prelaunch_activated==1].success.mean()*100:.0f}% of the time, those without {E[E.prelaunch_activated==0].success.mean()*100:.0f}%;\na video changes little",'Presentation','5_presentation',{0,1},horizontal=True)

### 6.6 Market

In [ ]:
k=pd.cut(E.cat_rate_12m,[-.01,.3,.45,.6,.75,.9,1.01],labels=['under 30%','30–45%','45–60%','60–75%','75–90%','over 90%'])
t=tab(k); bars(t,f"In categories where under 30% of recent campaigns were funded, {t.funded.iloc[0]*100:.0f}% of new ones are;\nwhere over 90% were, {t.funded.iloc[-1]*100:.0f}%",'Share of campaigns funded in the same category, among those ended in the 12 months before','6_market',{0})

## 7) Conclusions
1. 35 features in 6 groups, all computed from information available before launch; the history check finds no mismatch.
2. Strongest signals inside a category: creator track record, goal size, preparation time, pre-launch page.
3. Video alone and most pitch-wording features are weak on their own; TF-IDF in the model notebook will test the text properly.
4. Market features are partly an artefact of scraper coverage and must be read with care.
5. All rates here are sample rates: too high in absolute terms.